# 1.0 - Khảo sát ban đầu dữ liệu giá xăng dầu

Notebook này ghi lại lần xem dữ liệu thô đầu tiên, **trước khi** viết các lớp trong `src/gia_xang_dau`.
Mục đích: biết từng nguồn trông ra sao, có khuyết tật gì, từ đó quyết định các bước làm sạch.

- Notebook chỉ để khám phá. Kết quả nộp bài sinh ra từ `python chay.py --giai-doan tat-ca`.
- Cần có dữ liệu thô trong `data/raw` và `data/external` (chạy `python chay.py --giai-doan nap` nếu chưa có).
- Notebook chỉ đọc, không ghi tệp nào.

In [1]:
import html
import json
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)
GOC = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DU_LIEU = GOC / "data"
for tep in sorted(DU_LIEU.glob("*/*/*")):
    if "tim_kiem" not in tep.name:
        print(f"{str(tep.relative_to(GOC)):60s} {tep.stat().st_size / 1e6:8.2f} MB")
print("so trang tim kiem da luu:", len(list(DU_LIEU.glob("raw/*/tim_kiem*"))))

data/external/gpr/data_gpr_daily_recent.xls                      3.28 MB
data/external/oxcgrt/OxCGRT_compact_national_v1.csv             43.84 MB
data/raw/bcp/bai_viet.jsonl                                      2.66 MB
data/raw/eia/gia_giao_ngay_trang_001.json                        1.65 MB
data/raw/eia/gia_giao_ngay_trang_002.json                        1.65 MB
data/raw/eia/gia_giao_ngay_trang_003.json                        1.65 MB
data/raw/eia/gia_giao_ngay_trang_004.json                        1.65 MB
data/raw/eia/gia_giao_ngay_trang_005.json                        1.65 MB
data/raw/eia/gia_giao_ngay_trang_006.json                        1.65 MB
data/raw/eia/gia_giao_ngay_trang_007.json                        0.91 MB
data/raw/moit/bai_viet.jsonl                                     4.77 MB
data/raw/ty_gia/ty_gia_trang_001.json                            0.02 MB
data/raw/ty_gia/ty_gia_trang_002.json                            0.02 MB
data/raw/ty_gia/ty_gia_trang_003.json              

## 1. EIA - giá giao ngay (API REST phân trang)

Mỗi trang là một tệp JSON 5.000 bản ghi. Xem cấu trúc một trang và kiểu của từng trường.

In [2]:
trang_1 = json.loads((DU_LIEU / "raw/eia/gia_giao_ngay_trang_001.json").read_text(encoding="utf-8"))
print({khoa: (gia_tri if khoa != "data" else f"[{len(gia_tri)} ban ghi]") for khoa, gia_tri in trang_1["response"].items()})
eia = pd.concat(
    [pd.DataFrame(json.loads(tep.read_text(encoding="utf-8"))["response"]["data"])
     for tep in sorted((DU_LIEU / "raw/eia").glob("gia_giao_ngay_trang_*.json"))],
    ignore_index=True,
)
print(eia.shape)
eia[["period", "series", "value", "units"]].head()

{'total': '32760', 'dateFormat': 'YYYY-MM-DD', 'frequency': 'daily', 'data': '[5000 ban ghi]'}
(32760, 11)


,period,series,value,units
0,2015-01-02,EER_EPD2DC_PF4_Y05LA_DPG,1.704,$/GAL
1,2015-01-02,EER_EPD2DXL0_PF4_RGC_DPG,1.589,$/GAL
2,2015-01-02,EER_EPD2DXL0_PF4_Y35NY_DPG,1.805,$/GAL
3,2015-01-02,EER_EPD2F_PF4_Y35NY_DPG,1.709,$/GAL
4,2015-01-02,EER_EPJK_PF4_RGC_DPG,1.559,$/GAL


In [3]:
print(eia[["period", "value"]].dtypes.to_dict(), "| vi du:", eia["value"].iloc[:3].tolist())
print("khoang ngay:", eia["period"].min(), "->", eia["period"].max())
eia.groupby(["series", "units"]).agg(so_dong=("value", "size"), ngay_dau=("period", "min"), ngay_cuoi=("period", "max"))

{'period': dtype('O'), 'value': dtype('O')} | vi du: ['1.704', '1.589', '1.805']
khoang ngay: 2015-01-02 -> 2026-11-03


,,so_dong,ngay_dau,ngay_cuoi
series,units,,,
EER_EPD2DC_PF4_Y05LA_DPG,$/GAL,2972,2015-01-02,2026-11-03
EER_EPD2DXL0_PF4_RGC_DPG,$/GAL,2976,2015-01-02,2026-11-03
EER_EPD2DXL0_PF4_Y35NY_DPG,$/GAL,2976,2015-01-02,2026-11-03
EER_EPD2F_PF4_Y35NY_DPG,$/GAL,2976,2015-01-02,2026-11-03
EER_EPJK_PF4_RGC_DPG,$/GAL,2977,2015-01-02,2026-11-03
EER_EPLLPA_PF4_Y44MB_DPG,$/GAL,2973,2015-01-02,2026-11-03
EER_EPMRR_PF4_Y05LA_DPG,$/GAL,2976,2015-01-02,2026-11-03
EER_EPMRU_PF4_RGC_DPG,$/GAL,2976,2015-01-02,2026-11-03
EER_EPMRU_PF4_Y35NY_DPG,$/GAL,2976,2015-01-02,2026-11-03


**Nhận xét.** `value` là **chuỗi**, không phải số. Có 11 chuỗi với hai đơn vị (`$/BBL`, `$/GAL`) nên phải quy về một đơn vị trước khi so sánh.
Kiểm tra thêm ba thứ: ô thiếu, số viết không có số 0 đứng đầu, và giá âm.

In [4]:
gia = pd.to_numeric(eia["value"], errors="coerce")
print("o value thieu hoac khong doc duoc:", int(gia.isna().sum()))
print("so viet khong co so 0 dung dau (vi du):", eia.loc[eia["value"].str.startswith("."), "value"].head(3).tolist(),
      "| so o:", int(eia["value"].str.startswith(".").sum()))
print("ban ghi trung (period, series):", int(eia.duplicated(["period", "series"]).sum()))
eia.loc[gia < 0, ["period", "series", "value", "units"]]

o value thieu hoac khong doc duoc: 0
so viet khong co so 0 dung dau (vi du): ['.49', '.465', '.449'] | so o: 3181
ban ghi trung (period, series): 0


,period,series,value,units
14792,2020-04-20,RWTC,-36.98,$/BBL


**Nhận xét.** Giá WTI ngày 20/4/2020 là số âm. Đây là giá thật (hợp đồng tháng 5/2020 hết hạn giữa lúc kho chứa đầy), không phải lỗi nhập liệu,
nên được **giữ lại**; nhưng không lấy được logarit của giá âm, nên phép tính độ biến động phải bỏ hai phiên liên quan.

## 2. Bộ Công Thương - thông báo điều hành giá (bài viết HTML)

Giá bán lẻ không có sẵn dạng bảng. Mỗi kỳ điều hành là một bài viết; giá nằm trong câu văn.

In [5]:
def doc_jsonl(tep):
    return [json.loads(dong) for dong in tep.read_text(encoding="utf-8").splitlines() if dong.strip()]


def van_ban_thuan(ma_html):
    van_ban = re.sub(r"<br\s*/?>|</(?:p|div|li|tr|h[1-6])>", "\n", ma_html or "")
    return re.sub(r"[ \t\xa0]+", " ", html.unescape(re.sub(r"<[^>]+>", "", van_ban)))


moit = doc_jsonl(DU_LIEU / "raw/moit/bai_viet.jsonl")
print("so bai:", len(moit), "| cac truong:", list(moit[0]))
bai = moit[0]
print(bai["title"], "| publishTime =", bai["publishTime"])
for dong in van_ban_thuan(bai["content"]).splitlines():
    if re.search(r"không cao hơn|USD/thùng|[Áá]p dụng từ", dong):
        print("  ", dong.strip()[:150])

so bai: 306 | cac truong: ['id', 'type', 'title', 'brief', 'content', 'publishTime', 'createdTime', 'lastUpdateTime', 'categories', 'rewriteURL']
Một số thông tin về việc điều hành giá xăng dầu ngày 5/11 | publishTime = 1793866860
   Giá thành phẩm xăng dầu thế giới bình quân giữa kỳ điều hành giá ngày 03/11/2026 và kỳ điều hành ngày 05/11/2026 là: 112,615 USD/thùng xăng RON92 dùng
   Giá bán xăng dầu của các nước lân cận Việt Nam ngày 05/11/2026: Về giá xăng: Thái Lan: 32.001 VNĐ/lít (Chính phủ trợ giá); Campuchia: 31.887 VNĐ/lít (
   - Xăng E5RON92: không cao hơn 20.898 đồng/lít (giảm 2.305 đồng/lít so với giá bán tối đa hiện hành), thấp hơn xăng E10RON95-III 258 đồng/lít;
   - Xăng E10RON95-III: không cao hơn 21.156 đồng/lít (giảm 2.075 đồng/lít so với giá bán tối đa hiện hành);
   - Dầu điêzen 0.05S: không cao hơn 28.696 đồng/lít (giảm 180 đồng/lít so với giá bán tối đa hiện hành);
   - Dầu madút 180CST 3.5S: không cao hơn 22.005 đồng/kg (giảm 1.969 đồng/kg so với giá bán tối đa hi

**Nhận xét.**
- Số viết kiểu Việt Nam: `21.156` nghĩa là 21156 đồng; `80,560` nghĩa là 80,56 USD. Đọc thẳng bằng `float` sẽ sai một nghìn lần.
- Ngày viết bằng chữ: "15 giờ 00’ ngày 05 tháng 11 năm 2026"; `publishTime` lại là số giây Unix.
- Mỗi câu còn cho biết mức tăng/giảm so với giá hiện hành: dùng được để biết giá của ngay trước kỳ điều chỉnh.

Xem tên mặt hàng được viết theo bao nhiêu kiểu trong toàn bộ các bài.

In [6]:
mau = re.compile(r"((?:[Xx]ăng|[Dd]ầu)[^:;\n,]{0,40}?)\s*:?\s*[Kk]hông cao hơn\s+(\d{1,3}(?:[.,]\d{3})*)\s*(đồng/(?:lít|kg))")
nhan = pd.Series([khop[0].strip() for bai in moit for khop in mau.findall(van_ban_thuan(bai["content"]))])
nhan.value_counts().head(14)

Dầu hỏa                               272
Dầu mazut 180CST 3.5S                 209
Xăng E5RON92                          203
Dầu diesel 0.05S                      159
Dầu điêzen 0.05S                      126
Dầu madút 180CST 3.5S                  75
Xăng RON 92                            67
Xăng E5                                67
Xăng E5 RON92                          18
Xăng E10RON95-III                      13
xăng sinh học E5 giá bán                2
dầu mazut và ở mức                      1
xăng E5RON92                            1
xăng RON 92 tăng 306 đồng/lít lên      1
Name: count, dtype: int64

**Nhận xét.** Cùng một mặt hàng có nhiều cách viết ("Dầu điêzen 0.05S", "Dầu diesel 0.05S", "Xăng E5RON92", "Xăng E5 RON92"...),
và mặt hàng chuẩn thay đổi theo thời gian (RON 92 trước 2018; E10RON95-III từ giữa 2026). Cần bảng quy tắc chuẩn hoá danh mục.

## 3. Báo Chính phủ - bài báo về các kỳ điều hành

Dùng để lấp các kỳ mà trang Bộ Công Thương không có bài. Kiểm tra mức trùng với nguồn trên.

In [7]:
bcp = doc_jsonl(DU_LIEU / "raw/bcp/bai_viet.jsonl")
print("so bai:", len(bcp), "| cac truong:", list(bcp[0]))
print("vi du publishTime:", bcp[0]["publishTime"])
ngay_bcp = {bai["publishTime"][:10] for bai in bcp}
ngay_moit = {pd.Timestamp(int(bai["publishTime"]), unit="s", tz="Asia/Ho_Chi_Minh").strftime("%Y-%m-%d") for bai in moit}
print("so ngay co bai o ca hai nguon:", len(ngay_bcp & ngay_moit), "| chi bao co:", len(ngay_bcp - ngay_moit), "| chi bo co:", len(ngay_moit - ngay_bcp))
cau = [dong.strip() for bai in bcp[:40] for dong in van_ban_thuan(bai["content"]).splitlines() if "không cao hơn" in dong]
for dong in cau[:3] + [dong for dong in cau if re.search(r"(?:tăng|giảm) [\d.]+ đồng/lít, giá", dong)][:2]:
    print("  ", dong[:170])

so bai: 346 | cac truong: ['id', 'title', 'publishTime', 'content']
vi du publishTime: 2026-11-05T14:50:00+07:00
so ngay co bai o ca hai nguon: 205 | chi bao co: 128 | chi bo co: 100
   Giá xăng E5RON92 không cao hơn 20.898 đồng/lít (giảm 2.305 đồng/lít so với giá bán tối đa hiện hành), thấp hơn xăng E10RON95-III 258 đồng/lít.
   Giá xăng E10RON95-III không cao hơn 21.156 đồng/lít (giảm 2.075 đồng/lít so với giá bán tối đa hiện hành).
   Giá dầu diesel 0.05S không cao hơn 28.696 đồng/lít (giảm 180 đồng/lít so với giá bán tối đa hiện hành).


**Nhận xét.** Hai nguồn trùng nhau ở nhiều kỳ nên sẽ có **bản ghi trùng** theo khoá (ngày, mặt hàng): phải chọn nguồn ưu tiên.
Báo còn dùng một cách viết thứ hai ("X giảm N đồng/lít, giá bán không cao hơn P đồng/lít") và hay nhắc lại giá kỳ trước ở cuối bài.

## 4. Tỷ giá, chỉ số GPR và OxCGRT (ba bảng tra cứu)

In [8]:
ty_gia = pd.concat(
    [pd.DataFrame(json.loads(tep.read_text(encoding="utf-8"))["data"]) for tep in sorted((DU_LIEU / "raw/ty_gia").glob("*.json"))],
    ignore_index=True,
)
print(ty_gia.shape, "|", ty_gia["effective_date"].min(), "->", ty_gia["effective_date"].max())
ty_gia[["effective_date", "country_currency_desc", "exchange_rate"]].tail(4)

(103, 13) | 2001-03-31 -> 2026-09-30


,effective_date,country_currency_desc,exchange_rate
99,2025-12-31,Vietnam-Dong,26295.0
100,2026-03-31,Vietnam-Dong,26323.0
101,2026-06-30,Vietnam-Dong,26255.0
102,2026-09-30,Vietnam-Dong,25959.0


In [9]:
gpr = pd.read_excel(DU_LIEU / "external/gpr/data_gpr_daily_recent.xls")
print(gpr.shape, "| cot:", list(gpr.columns))
print("kieu cot DAY:", gpr["DAY"].dtype, "| vi du:", gpr["DAY"].iloc[-2:].tolist(), "| GPRD thieu:", int(gpr["GPRD"].isna().sum()))
gpr[["DAY", "GPRD"]].tail(3)

(15281, 11) | cot: ['DAY', 'N10D', 'GPRD', 'GPRD_ACT', 'GPRD_THREAT', 'date', 'GPRD_MA30', 'GPRD_MA7', 'event', 'var_name', 'var_label']
kieu cot DAY: int64 | vi du: [20261101, 20261102] | GPRD thieu: 0


,DAY,GPRD
15278,20261031,180.685516
15279,20261101,86.264778
15280,20261102,113.945816


In [10]:
tep_ox = DU_LIEU / "external/oxcgrt/OxCGRT_compact_national_v1.csv"
print(f"kich thuoc tep: {tep_ox.stat().st_size / 1e6:.1f} MB")
ox = pd.read_csv(tep_ox, usecols=["CountryCode", "Jurisdiction", "Date", "StringencyIndex_Average"], dtype=str)
print(ox.shape, "| Jurisdiction:", ox["Jurisdiction"].value_counts().to_dict())
print("khoang ngay:", ox["Date"].min(), "->", ox["Date"].max(), "| so nuoc:", ox["CountryCode"].nunique(),
      "| o chi so thieu:", int(ox["StringencyIndex_Average"].isna().sum()))
ox.head(3)

kich thuoc tep: 43.8 MB


(202760, 4) | Jurisdiction: {'NAT_TOTAL': 202760}
khoang ngay: 20200101 -> 20221231 | so nuoc: 185 | o chi so thieu: 0


,CountryCode,Jurisdiction,Date,StringencyIndex_Average
0,ABW,NAT_TOTAL,20200101,0.00
1,ABW,NAT_TOTAL,20200102,0.00
2,ABW,NAT_TOTAL,20200103,0.00


**Nhận xét.**
- Tỷ giá chỉ có **mỗi quý một giá trị** trong khi giá có theo ngày: khoá `ngay` không khớp, phải nội suy.
- GPR và OxCGRT ghi ngày dạng số `yyyymmdd` (20200115), khác dạng `yyyy-mm-dd` của EIA.
- OxCGRT dừng ở 31/12/2022: sau mốc đó không có chỉ số phong toả, phải để thiếu chứ không được tự điền.
- Tệp OxCGRT hơn 40 MB với 56 cột, chỉ cần 4 cột: đọc theo khối và chỉ lấy cột cần dùng.

## 5. Tổng kết khuyết tật và việc cần làm

| Khuyết tật thấy được | Ở nguồn nào | Việc cần làm trong mã |
|---|---|---|
| Số và ngày đều là chuỗi, mỗi nguồn một kiểu viết | mọi nguồn | lớp chuyển kiểu ngày (nhiều mẫu) và kiểu số (kiểu Việt Nam / kiểu chuẩn) |
| Tên mặt hàng viết nhiều kiểu, đổi theo thời gian | Bộ Công Thương, báo | bảng quy tắc chuẩn hoá danh mục trong `cau_hinh.yaml` |
| Cùng một giá xuất hiện ở hai nguồn và nhiều lần trong một bài | Bộ Công Thương, báo | bỏ bản ghi trùng theo khoá (ngày, mặt hàng), có thứ tự ưu tiên nguồn |
| Bảng tra cứu không có giá trị đúng ngày quan sát | tỷ giá, GPR, OxCGRT | ghép theo khoá `ngay`, điền theo quy tắc riêng của từng cột và đếm số ô đã điền |
| Thiếu bài của một số kỳ điều hành | Bộ Công Thương, báo | dùng mức tăng/giảm nêu trong câu để suy ra giá ngay trước kỳ điều chỉnh; đếm số khoảng còn thiếu |
| Giá âm (WTI 20/4/2020) | EIA | giữ lại vì là giá thật; loại khỏi phép tính logarit |